# CS5480 — Deep Learning Kaggle Competition
Run each cell top-to-bottom. Make sure **Runtime → Change runtime type → GPU** is selected.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Set Data Path
Edit `DATA_DIR` to point to your dataset folder inside Drive.
It must contain `train/0/`, `train/1/`, and `test/`.

In [ ]:
DATA_DIR = '/content/drive/MyDrive/data'  # <-- change this if needed

## 3. Verify GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 4. Imports & Config

In [ ]:
import math
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import torchvision.models as tvm
from PIL import Image
from pathlib import Path
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader, Subset


@dataclass
class CFG:
    seed: int = 42
    img_size: int = 224
    batch_size: int = 128
    num_workers: int = 2       # keep low in Colab
    epochs: int = 80
    lr: float = 7e-4
    weight_decay: float = 1e-4
    warmup_epochs: int = 5
    dropout: float = 0.3
    val_split: float = 0.15
    label_smoothing: float = 0.0
    grad_clip: float = 1.0
    tta: bool = False
    patience: int = 15
    mixup_alpha: float = 0.4

## 5. Utilities (seed, transforms, GeM, Mixup)

In [ ]:
def set_seed(*, seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_transforms(cfg: CFG, train: bool):
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]
    if train:
        return T.Compose([
            T.RandomResizedCrop(cfg.img_size, scale=(0.7, 1.0)),
            T.RandomHorizontalFlip(),
            T.ColorJitter(0.4, 0.4, 0.4, 0.15),
            T.RandomApply([T.GaussianBlur(3)], p=0.2),
            T.ToTensor(),
            T.Normalize(mean, std),
        ])
    else:
        return T.Compose([
            T.Resize((cfg.img_size, cfg.img_size)),
            T.ToTensor(),
            T.Normalize(mean, std),
        ])


class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return torch.nn.functional.adaptive_avg_pool2d(
            x.clamp(min=self.eps).pow(self.p), (1, 1)
        ).pow(1.0 / self.p)


def mixup_batch(xb, yb, alpha: float):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(xb.size(0), device=xb.device)
    mixed_x = lam * xb + (1 - lam) * xb[idx]
    return mixed_x, yb, yb[idx], lam

## 6. Model (ResNet50, no pretrained weights)

In [ ]:
class Model(nn.Module):
    def __init__(self, *, cfg: CFG):
        super().__init__()
        backbone = tvm.resnet50(weights=None)
        self.features = nn.Sequential(*list(backbone.children())[:-2])
        self.pool = GeM()
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(cfg.dropout),
            nn.Linear(2048, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(cfg.dropout / 2),
            nn.Linear(256, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)

## 7. Dataset & Scheduler

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, *, root_dir: Path, cfg: CFG, train: bool):
        self.samples = []
        self.transform = get_transforms(cfg, train)
        for label in ['0', '1']:
            class_dir = Path(root_dir) / label
            for path in sorted(class_dir.iterdir()):
                if path.is_file():
                    self.samples.append((path, int(label)))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        x = self.transform(img)
        y = torch.tensor([label], dtype=torch.float32)
        return x, y


def get_scheduler(optimizer, cfg):
    def lr_lambda(epoch):
        if epoch < cfg.warmup_epochs:
            return (epoch + 1) / cfg.warmup_epochs
        progress = (epoch - cfg.warmup_epochs) / max(cfg.epochs - cfg.warmup_epochs, 1)
        return 0.5 * (1 + math.cos(math.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

## 8. Evaluate & Threshold Search

In [ ]:
def evaluate(*, model, loader, device):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = (torch.sigmoid(model(xb)) > 0.5).float()
            correct += (preds == yb).sum().item()
            total += yb.numel()
    return correct / total


def find_best_threshold(*, model, loader, device):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            p = torch.sigmoid(model(xb)).cpu().numpy().flatten()
            all_probs.extend(p)
            all_labels.extend(yb.numpy().flatten())
    probs  = np.array(all_probs)
    labels = np.array(all_labels)
    best_t, best_acc = 0.5, 0.0
    for t in np.linspace(0.05, 0.95, 401):
        acc = ((probs > t).astype(float) == labels).mean()
        if acc > best_acc:
            best_acc = acc
            best_t = float(t)
    print(f'Best threshold: {best_t:.3f}  Val acc: {best_acc:.4f}')
    return best_t

## 9. Training Loop

In [ ]:
def _run_epochs(*, model, loader, cfg, device, show_val=False, val_loader=None):
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = get_scheduler(optimizer, cfg)
    scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')
    smooth = cfg.label_smoothing

    best_val_acc = 0.0
    best_state = None
    no_improve = 0

    for epoch in range(cfg.epochs):
        model.train()
        total_loss = correct = total = 0

        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)

            if show_val and cfg.mixup_alpha > 0:
                xb, ya, yb_mix, lam = mixup_batch(xb, yb, cfg.mixup_alpha)
                ya_s = ya    * (1 - smooth) + (1 - ya)    * (smooth / 2)
                yb_s = yb_mix * (1 - smooth) + (1 - yb_mix) * (smooth / 2)
            else:
                ya_s = yb * (1 - smooth) + (1 - yb) * (smooth / 2)
                yb_s, lam = ya_s, 1.0

            optimizer.zero_grad()
            with torch.amp.autocast('cuda', enabled=device.type == 'cuda'):
                logits = model(xb)
                loss = (lam * nn.functional.binary_cross_entropy_with_logits(logits, ya_s)
                        + (1 - lam) * nn.functional.binary_cross_entropy_with_logits(logits, yb_s))

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()
            correct += ((torch.sigmoid(logits) > 0.5).float() == yb).sum().item()
            total += yb.numel()

        scheduler.step()

        if show_val and val_loader is not None:
            val_acc = evaluate(model=model, loader=val_loader, device=device)
            print(f'Epoch {epoch+1}/{cfg.epochs} | Loss: {total_loss/len(loader):.4f} | '
                  f'Train Acc: {correct/total:.4f} | Val Acc: {val_acc:.4f}')
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= cfg.patience:
                    print(f'Early stopping at epoch {epoch+1}')
                    break
        else:
            print(f'[Full] Epoch {epoch+1}/{cfg.epochs} | Loss: {total_loss/len(loader):.4f}')

    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
        print(f'Restored best model (val acc: {best_val_acc:.4f})')

## 10. Train / Predict / Submit

In [ ]:
def train_with_validation(*, model, train_dir, cfg):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    train_ds = ImageDataset(root_dir=train_dir, cfg=cfg, train=True)
    val_ds   = ImageDataset(root_dir=train_dir, cfg=cfg, train=False)
    g = torch.Generator().manual_seed(cfg.seed)
    indices = torch.randperm(len(train_ds), generator=g).tolist()
    split = int((1 - cfg.val_split) * len(train_ds))
    train_idx, val_idx = indices[:split], indices[split:]
    train_loader = DataLoader(Subset(train_ds, train_idx), batch_size=cfg.batch_size,
                              shuffle=True, num_workers=cfg.num_workers, pin_memory=True,
                              persistent_workers=cfg.num_workers > 0)
    val_loader   = DataLoader(Subset(val_ds, val_idx), batch_size=cfg.batch_size,
                              shuffle=False, num_workers=cfg.num_workers, pin_memory=True,
                              persistent_workers=cfg.num_workers > 0)
    _run_epochs(model=model, loader=train_loader, cfg=cfg, device=device,
                show_val=True, val_loader=val_loader)
    threshold = find_best_threshold(model=model, loader=val_loader, device=device)
    return model, threshold


def train_full(*, model, train_dir, cfg):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    dataset = ImageDataset(root_dir=train_dir, cfg=cfg, train=True)
    loader  = DataLoader(dataset, batch_size=cfg.batch_size, shuffle=True,
                         num_workers=cfg.num_workers, pin_memory=True,
                         persistent_workers=cfg.num_workers > 0)
    _run_epochs(model=model, loader=loader, cfg=cfg, device=device)
    return model


def predict(*, model, test_dir, cfg, threshold):
    model.eval()
    device = next(model.parameters()).device
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]
    normalize = T.Compose([T.ToTensor(), T.Normalize(mean, std)])
    tta_fns = [
        lambda img: img,
        TF.hflip,
        TF.vflip,
        lambda img: TF.hflip(TF.vflip(img)),
    ]
    def prep(img):
        img = img.resize((cfg.img_size, cfg.img_size), Image.BILINEAR)
        return normalize(img).unsqueeze(0).to(device)
    results = []
    for path in sorted(Path(test_dir).iterdir()):
        if not path.is_file():
            continue
        img = Image.open(path).convert('RGB')
        with torch.no_grad():
            if cfg.tta:
                prob = sum(torch.sigmoid(model(prep(fn(img)))).item() for fn in tta_fns) / len(tta_fns)
            else:
                prob = torch.sigmoid(model(prep(img))).item()
        results.append((path.name, int(prob > threshold)))
    return results

## 11. Run Pipeline

In [ ]:
cfg = CFG()
set_seed(seed=cfg.seed)

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

data_dir  = Path(DATA_DIR)
train_dir = data_dir / 'train'
test_dir  = data_dir / 'test'

print('=== Training with validation ===')
model = Model(cfg=cfg)
model, threshold = train_with_validation(model=model, train_dir=train_dir, cfg=cfg)

print('=== Retraining on full dataset ===')
model = Model(cfg=cfg)
model = train_full(model=model, train_dir=train_dir, cfg=cfg)

print('=== Predicting ===')
results = predict(model=model, test_dir=test_dir, cfg=cfg, threshold=threshold)

df = pd.DataFrame(results, columns=['ID', 'TARGET'])
df.to_csv('submission.csv', index=False)
print(f'submission.csv saved — {len(df)} rows')
print(df.head())

## 12. Copy submission.csv to Drive & Submit to Kaggle (optional)

In [ ]:
import shutil

# Save a copy to Drive so it survives session resets
drive_out = Path('/content/drive/MyDrive/submission.csv')
shutil.copy('submission.csv', drive_out)
print(f'Copied to {drive_out}')

In [ ]:
# --- Kaggle submission (run only if kaggle.json is in Drive) ---
# Copy your kaggle.json to Colab first:
import os
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
shutil.copy('/content/drive/MyDrive/kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

!pip install -q kaggle
!kaggle competitions submit \
    -c iith-deep-learning-2026-hackathon \
    -f submission.csv \
    -m "resnet50-gem-mixup"